# Download DWD Historical Hourly Air Temperature Data

This notebook downloads all zip files from the DWD (Deutscher Wetterdienst) historical hourly air temperature data repository.

Source: https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/air_temperature/historical/


In [ ]:
!uv pip install bs4 tqdm requests


Using Python 3.11.14 environment at: /home/abhishek/Documents/heatpump_ai_2/climate_data_prep/.venv
Audited 3 packages in 8ms


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from urllib.parse import urljoin, unquote
from pathlib import Path
import time
from tqdm import tqdm
import zipfile


In [ ]:
# Configuration
BASE_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/air_temperature/historical/"
OUTPUT_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature")

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Directory exists: {OUTPUT_DIR.exists()}")


Output directory: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature
Directory exists: True


In [ ]:
def get_zip_files_from_directory(url):
    """
    Parse the directory listing page and extract all zip file URLs.
    
    Args:
        url: URL to the directory listing page
        
    Returns:
        List of zip file URLs
    """
    response = requests.get(url)
    response.raise_for_status()
    
    # Parse HTML
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find all links that end with .zip
    zip_files = []
    for link in soup.find_all('a'):
        href = link.get('href', '')
        if href.endswith('.zip'):
            # Decode URL-encoded filename
            filename = unquote(href)
            full_url = urljoin(url, href)
            zip_files.append((filename, full_url))
    
    return zip_files


In [ ]:
def download_file(url, output_path, chunk_size=8192):
    """
    Download a file from URL to output path with progress bar.
    
    Args:
        url: URL to download from
        output_path: Path where file should be saved
        chunk_size: Size of chunks to read at a time
        
    Returns:
        True if successful, False otherwise
    """
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(output_path, 'wb') as f, tqdm(
            desc=output_path.name,
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
        ) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
        
        return True
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False


In [ ]:
# Get list of all zip files
print("Fetching list of zip files from DWD server...")
zip_files = get_zip_files_from_directory(BASE_URL)
print(f"Found {len(zip_files)} zip files")
print(f"\nFirst 5 files:")
for filename, url in zip_files[:5]:
    print(f"  - {filename}")


Fetching list of zip files from DWD server...
Found 636 zip files

First 5 files:
  - stundenwerte_TU_00003_19500401_20110331_hist.zip
  - stundenwerte_TU_00044_20070401_20241231_hist.zip
  - stundenwerte_TU_00052_19760101_19880101_hist.zip
  - stundenwerte_TU_00071_20091201_20191231_hist.zip
  - stundenwerte_TU_00073_20070401_20241231_hist.zip


In [ ]:
# Download all zip files
print(f"\nStarting download of {len(zip_files)} files to {OUTPUT_DIR}...")
print("=" * 80)

downloaded = 0
skipped = 0
failed = 0

for filename, url in zip_files:
    output_path = OUTPUT_DIR / filename
    
    # Skip if file already exists
    if output_path.exists():
        print(f"Skipping {filename} (already exists)")
        skipped += 1
        continue
    
    print(f"\nDownloading: {filename}")
    success = download_file(url, output_path)
    
    if success:
        downloaded += 1
        # Small delay to be respectful to the server
        time.sleep(0.5)
    else:
        failed += 1

print("\n" + "=" * 80)
print(f"Download complete!")
print(f"  Downloaded: {downloaded}")
print(f"  Skipped (already exists): {skipped}")
print(f"  Failed: {failed}")
print(f"  Total: {len(zip_files)}")



Starting download of 636 files to /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature...

Downloading: stundenwerte_TU_00003_19500401_20110331_hist.zip


stundenwerte_TU_00003_19500401_20110331_hist.zip: 100%|██████████| 2.73M/2.73M [00:01<00:00, 2.76MB/s]



Downloading: stundenwerte_TU_00044_20070401_20241231_hist.zip


stundenwerte_TU_00044_20070401_20241231_hist.zip: 100%|██████████| 816k/816k [00:00<00:00, 999kB/s] 



Downloading: stundenwerte_TU_00052_19760101_19880101_hist.zip


...


Downloading: stundenwerte_TU_19207_20230401_20241231_hist.zip


stundenwerte_TU_19207_20230401_20241231_hist.zip: 100%|██████████| 88.1k/88.1k [00:00<00:00, 291kB/s]



Downloading: stundenwerte_TU_19856_20240801_20241231_hist.zip


stundenwerte_TU_19856_20240801_20241231_hist.zip: 100%|██████████| 24.9k/24.9k [00:00<00:00, 274kB/s]



Download complete!
  Downloaded: 636
  Skipped (already exists): 0
  Failed: 0
  Total: 636


In [13]:
# Verify downloaded files
print(f"\nVerifying downloaded files in {OUTPUT_DIR}...")
downloaded_files = list(OUTPUT_DIR.glob("*.zip"))
print(f"Found {len(downloaded_files)} zip files in output directory")

if len(downloaded_files) > 0:
    print(f"\nTotal size: {sum(f.stat().st_size for f in downloaded_files) / (1024**3):.2f} GB")
    print(f"\nSample files:")
    for f in downloaded_files[:5]:
        size_mb = f.stat().st_size / (1024**2)
        print(f"  - {f.name} ({size_mb:.2f} MB)")



Verifying downloaded files in /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature...
Found 636 zip files in output directory

Total size: 0.76 GB

Sample files:
  - stundenwerte_TU_00003_19500401_20110331_hist.zip (2.73 MB)
  - stundenwerte_TU_00044_20070401_20241231_hist.zip (0.80 MB)
  - stundenwerte_TU_00052_19760101_19880101_hist.zip (0.53 MB)
  - stundenwerte_TU_00071_20091201_20191231_hist.zip (0.47 MB)
  - stundenwerte_TU_00073_20070401_20241231_hist.zip (0.80 MB)


In [14]:
# Compare server files with downloaded files and identify missing ones
print("=" * 80)
print("Checking for missing files...")
print("=" * 80)

# Get list of files from server
print("\nFetching list of files from DWD server...")
server_files = get_zip_files_from_directory(BASE_URL)
server_filenames = {filename for filename, url in server_files}
print(f"Found {len(server_filenames)} files on server")

# Get list of downloaded files
print(f"\nChecking downloaded files in {OUTPUT_DIR}...")
downloaded_files = list(OUTPUT_DIR.glob("*.zip"))
downloaded_filenames = {f.name for f in downloaded_files}
print(f"Found {len(downloaded_filenames)} files downloaded")

# Find missing files
missing_files = server_filenames - downloaded_filenames

print("\n" + "=" * 80)
print(f"COMPARISON RESULTS:")
print("=" * 80)
print(f"Files on server: {len(server_filenames)}")
print(f"Files downloaded: {len(downloaded_filenames)}")
print(f"Missing files: {len(missing_files)}")

if missing_files:
    print(f"\n{'=' * 80}")
    print("MISSING FILES LIST:")
    print("=" * 80)
    # Sort for easier reading
    missing_sorted = sorted(missing_files)
    for i, filename in enumerate(missing_sorted, 1):
        # Find the URL for this file
        url = next((url for fname, url in server_files if fname == filename), "N/A")
        print(f"{i:4d}. {filename}")
        if url != "N/A":
            print(f"      URL: {url}")
else:
    print("\n✓ All files have been successfully downloaded!")


Checking for missing files...

Fetching list of files from DWD server...
Found 636 files on server

Checking downloaded files in /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature...
Found 636 files downloaded

COMPARISON RESULTS:
Files on server: 636
Files downloaded: 636
Missing files: 0

✓ All files have been successfully downloaded!


In [16]:
# Extract all zip files
import zipfile  # Ensure zipfile is imported

print("=" * 80)
print("Extracting zip files...")
print("=" * 80)

# Create extraction directory (same name with _extracted suffix)
EXTRACTION_DIR = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_extracted"
EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extraction directory: {EXTRACTION_DIR}")

# Get all zip files
zip_files = list(OUTPUT_DIR.glob("*.zip"))
print(f"\nFound {len(zip_files)} zip files to extract")

extracted_count = 0
skipped_count = 0
failed_count = 0

for zip_file in tqdm(zip_files, desc="Extracting", unit="file"):
    try:
        # Create a subdirectory for each zip file's contents
        zip_extract_dir = EXTRACTION_DIR / zip_file.stem
        zip_extract_dir.mkdir(exist_ok=True)
        
        # Check if already extracted (by checking if directory has files)
        if zip_extract_dir.exists() and any(zip_extract_dir.iterdir()):
            skipped_count += 1
            continue
        
        # Extract the zip file
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(zip_extract_dir)
        
        extracted_count += 1
    except Exception as e:
        print(f"\nError extracting {zip_file.name}: {e}")
        failed_count += 1

print("\n" + "=" * 80)
print("Extraction complete!")
print("=" * 80)
print(f"  Extracted: {extracted_count}")
print(f"  Skipped (already extracted): {skipped_count}")
print(f"  Failed: {failed_count}")
print(f"  Total: {len(zip_files)}")
print(f"\nExtracted files location: {EXTRACTION_DIR}")


Extracting zip files...
Extraction directory: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted

Found 636 zip files to extract


Extracting:   0%|          | 0/636 [00:00<?, ?file/s]

Extracting: 100%|██████████| 636/636 [03:30<00:00,  3.03file/s]


Extraction complete!
  Extracted: 636
  Skipped (already extracted): 0
  Failed: 0
  Total: 636

Extracted files location: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted


In [ ]:
# Convert semicolon-separated txt files to CSV format
import pandas as pd
from pathlib import Path
import shutil

print("=" * 80)
print("Converting semicolon-separated txt files to CSV...")
print("=" * 80)

# Source and destination directories
SOURCE_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted")
DEST_DIR = Path("/mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv")

# Create destination directory
DEST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Source directory: {SOURCE_DIR}")
print(f"Destination directory: {DEST_DIR}")

# Get all subdirectories (one for each extracted zip file)
subdirs = [d for d in SOURCE_DIR.iterdir() if d.is_dir()]
print(f"\nFound {len(subdirs)} subdirectories to process")

# Try multiple encodings in order of likelihood
ENCODINGS = ['iso-8859-1', 'utf-8', 'windows-1252', 'cp1252']

def read_with_encoding(file_path, encodings):
    """Try to read a file with multiple encodings."""
    last_error = None
    for encoding in encodings:
        try:
            # Read with pandas using the specified encoding
            df = pd.read_csv(file_path, sep=';', encoding=encoding, low_memory=False, on_bad_lines='skip')
            return df
        except Exception as e:
            # Check if it's an encoding-related error
            error_type = type(e).__name__
            error_msg = str(e).lower()
            
            # Check for encoding-related exceptions or error messages
            is_encoding_error = (
                isinstance(e, (UnicodeDecodeError, UnicodeError)) or
                'codec' in error_msg or 
                'decode' in error_msg or 
                'encoding' in error_msg or
                'utf-8' in error_msg or
                'utf8' in error_msg or
                'invalid start byte' in error_msg or
                'invalid continuation byte' in error_msg
            )
            
            if is_encoding_error:
                last_error = e
                continue
            else:
                # For non-encoding errors, re-raise immediately
                raise
    # If we get here, all encodings failed
    raise ValueError(f"Could not decode {file_path} with any of the tried encodings. Last error: {last_error}")

converted_count = 0
skipped_count = 0
failed_count = 0
total_files = 0

for subdir in tqdm(subdirs, desc="Processing directories", unit="dir"):
    # Create corresponding subdirectory in destination
    dest_subdir = DEST_DIR / subdir.name
    dest_subdir.mkdir(exist_ok=True)
    
    # Find all txt files in this subdirectory
    txt_files = list(subdir.glob("*.txt"))
    total_files += len(txt_files)
    
    for txt_file in txt_files:
        try:
            # Destination CSV file path
            csv_file = dest_subdir / f"{txt_file.stem}.csv"
            
            # Skip if CSV already exists
            if csv_file.exists():
                skipped_count += 1
                continue
            
            # Read semicolon-separated file with proper encoding handling
            df = read_with_encoding(txt_file, ENCODINGS)
            
            # Write as CSV (comma-separated)
            df.to_csv(csv_file, index=False, encoding='utf-8')
            
            converted_count += 1
        except Exception as e:
            print(f"\nError converting {txt_file}: {e}")
            failed_count += 1

print("\n" + "=" * 80)
print("Conversion complete!")
print("=" * 80)
print(f"  Converted: {converted_count}")
print(f"  Skipped (already exists): {skipped_count}")
print(f"  Failed: {failed_count}")
print(f"  Total txt files found: {total_files}")
print(f"\nCSV files location: {DEST_DIR}")


Converting semicolon-separated txt files to CSV...
Source directory: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted
Destination directory: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv

Found 636 subdirectories to process


Processing directories: 100%|██████████| 636/636 [09:21<00:00,  1.13dir/s]


Conversion complete!
  Converted: 4336
  Skipped (already exists): 615
  Failed: 0
  Total txt files found: 4951

CSV files location: /mnt/d/heatpump_data/climate_data/dwd_historical_hourly_temperature_extracted_csv
